Idiopathic pulmonary fibrosis is a devastating condition, and analyzing a real-world bulk RNA-seq dataset for it provides a powerful, clinically relevant window into complex disease mechanisms. Breaking down the medical terminology reveals exactly what is happening at a tissue level, where "idiopathic" indicates that the underlying trigger remains unknown, "pulmonary" refers to the lungs, and "fibrosis" describes progressive scarring. This pathological scarring stiffens the lung tissue and severely impairs alveolar gas exchange, leading to the grim prognosis of a median survival of only two and a half to three and a half years post-diagnosis and a five-year survival rate between twenty and forty percent. Because there is currently no cure and treatments can only slow down disease progression rather than reverse it, early detection and a deep molecular understanding of the pathways driving fibrosis are critical.

The dataset structured for this project is exceptionally well-designed for a rigorous bioinformatics analysis. It comprises bulk RNA-seq data measuring the mRNA expression levels of 18,838 genes across 103 samples, balanced evenly between IPF conditions and non-IPF controls. This scale gives enough statistical power to detect meaningful transcriptomic shifts while covering a comprehensive set of human genes.

The composition of the samples reflects careful experimental design to account for biological and technical variance. On the disease side, the 103 IPF samples are split between 36 surgical biopsies taken from living patients with confirmed disease and 67 samples extracted from severely diseased lungs removed entirely during transplant surgeries. This captures both earlier-stage active fibrotic remodeling and end-stage architectural destruction.

Meanwhile, the 103 non-IPF control samples provide a robust baseline by incorporating two distinct sources. Forty-one samples come from histologically normal, non-cancerous margins of lung cancer biopsies, while the remaining 62 samples come from deceased organ donors without any history of lung disease. Including both tumor-adjacent normal tissue and healthy donor lungs allows researchers to control for potential confounding factors introduced by the biopsy procedure itself, ensuring that any differentially expressed genes discovered in the analysis truly reflect fibrotic disease processes rather than artifacts of tissue collection or co-morbidities.

In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ks_2samp

In [41]:
main_df = pd.read_csv("../data/GSE150910/GSE150910_gene-level_count_file.csv")
main_df.head()

,symbol,chp_26,chp_31,chp_34,chp_38,chp_1,chp_3,chp_4,chp_11,chp_5,...,chp_106,chp_108,chp_109,chp_110,chp_111,chp_112,chp_113,chp_114,ipf_1125,ipf_1130
0,TSPAN6,1361,993,351,613,841,565,1093,6239,1714,...,466,638,334,349,778,872,693,465,639,944
1,TNMD,5,13,0,0,0,6,3,4,0,...,0,0,0,0,0,0,0,0,0,0
2,DPM1,1929,2775,1894,2007,1436,1923,1956,4518,2411,...,936,1162,1043,689,1006,870,1701,860,1387,2150
3,SCYL3,176,216,208,218,162,137,242,490,259,...,57,35,122,42,73,68,145,21,146,204
4,C1orf112,93,143,97,148,98,128,102,188,119,...,50,60,72,21,74,48,131,72,69,158


In [42]:
genes = main_df['symbol'].values

In [43]:
set([column_name.split('_')[0] for column_name in main_df.columns.tolist()[1:]])

{'chp', 'control', 'ipf'}

In [44]:
df = main_df.copy()

In [45]:
df = df.filter(regex='control|ipf', axis=1)
df.head()

,ipf_724,ipf_726,ipf_729,ipf_730,ipf_731,ipf_732,ipf_733,ipf_735,control_738,ipf_739,...,control_1092,control_1096,control_1097,control_1102,control_1109,control_1111,control_1112,control_1113,ipf_1125,ipf_1130
0,849,1093,844,2918,1055,1063,3297,1632,2240,940,...,707,566,1338,577,655,1028,780,1092,639,944
1,0,0,0,9,0,3,0,0,0,0,...,0,0,0,0,0,5,0,6,0,0
2,2388,2229,1492,2464,2730,2360,2190,2390,2739,1711,...,2550,2115,2147,2045,2060,2841,2304,2084,1387,2150
3,165,217,148,363,278,328,287,249,330,170,...,335,220,206,213,158,216,257,199,146,204
4,104,161,103,188,212,153,104,143,248,112,...,156,79,96,87,83,192,143,114,69,158


In [46]:
# turn every row and column to float
df = df.apply(pd.to_numeric, axis=1,)
df.head()

,ipf_724,ipf_726,ipf_729,ipf_730,ipf_731,ipf_732,ipf_733,ipf_735,control_738,ipf_739,...,control_1092,control_1096,control_1097,control_1102,control_1109,control_1111,control_1112,control_1113,ipf_1125,ipf_1130
0,849,1093,844,2918,1055,1063,3297,1632,2240,940,...,707,566,1338,577,655,1028,780,1092,639,944
1,0,0,0,9,0,3,0,0,0,0,...,0,0,0,0,0,5,0,6,0,0
2,2388,2229,1492,2464,2730,2360,2190,2390,2739,1711,...,2550,2115,2147,2045,2060,2841,2304,2084,1387,2150
3,165,217,148,363,278,328,287,249,330,170,...,335,220,206,213,158,216,257,199,146,204
4,104,161,103,188,212,153,104,143,248,112,...,156,79,96,87,83,192,143,114,69,158


In [47]:
column_sums = df.sum()
df = df.apply(lambda x: (x / column_sums) * 1_000_000, axis=1)

In [48]:
df.head()

,ipf_724,ipf_726,ipf_729,ipf_730,ipf_731,ipf_732,ipf_733,ipf_735,control_738,ipf_739,...,control_1092,control_1096,control_1097,control_1102,control_1109,control_1111,control_1112,control_1113,ipf_1125,ipf_1130
0,26.202635,39.826077,32.456940,102.684442,23.479752,33.248302,107.388687,57.548092,58.371059,31.989574,...,18.778178,15.958816,34.102425,19.339868,27.895753,26.026071,27.463970,33.176158,31.235186,30.728696
1,0.000000,0.000000,0.000000,0.316710,0.000000,0.093833,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.126586,0.000000,0.182287,0.000000,0.000000
2,73.700699,81.218962,57.376486,86.708178,60.758032,73.815608,71.331885,84.276923,71.374254,58.227831,...,67.728931,59.634090,54.721903,68.544247,87.733207,71.926136,81.124341,63.314206,67.798439,69.985906
3,5.092385,7.906916,5.691501,12.773973,6.187082,10.259118,9.348060,8.780315,8.599308,5.785348,...,8.897722,6.203073,5.250448,7.139327,6.729052,5.468513,9.049026,6.045838,7.136678,6.640523
4,3.209746,5.866421,3.960977,6.615721,4.718206,4.785503,3.387450,5.042510,6.462510,3.811524,...,4.143417,2.227467,2.446811,2.916063,3.534882,4.860900,5.035061,3.463445,3.372814,5.143150


In [49]:
# get columns that only contain control
control_columns = [column for column in df.columns if 'control' in column]
# mean of every row for control columns
mean_CPM_control = df[control_columns].mean(axis=1)
mean_CPM_control[:5]

0    27.966734
1     0.333938
2    65.460376
3     7.022443
4     3.522556
dtype: float64

In [50]:
# get columns that only contain ipf
ipf_columns = [column for column in df.columns if 'ipf' in column]
# mean of every row for ipf columns
mean_CPM_ipf = df[ipf_columns].mean(axis=1)
mean_CPM_ipf[:5]

0    49.065821
1     0.125540
2    70.279478
3     7.140005
4     3.936511
dtype: float64

In [51]:
len(mean_CPM_control)

18838

In [52]:
to_keep = (mean_CPM_control >= 5) | (mean_CPM_ipf >= 5)
to_keep[:5]

0     True
1    False
2     True
3     True
4    False
dtype: bool

In [53]:
len(to_keep), to_keep.sum().item()

(18838, 11097)

In [54]:
df = df.loc[to_keep]
len(df)

11097

In [55]:
# df.iloc[0].filter(regex='control|ipf', axis=0).plot(kind='bar', figsize=(12, 6))
control = df.iloc[0].filter(regex='control', axis=0)
ipf = df.iloc[0].filter(regex='ipf', axis=0)
control.shape, ipf.shape

((103,), (103,))

In [56]:
ks_statics, p_value = ks_2samp(control, ipf)

print(f"KS Statistic: {ks_statics}")
print(f"P-value: {p_value}")

KS Statistic: 0.4368932038834951
P-value: 3.264129457102852e-09


In [57]:
# 1. Find matching column names ONCE outside the loop
control_cols = df.filter(regex='control').columns
ipf_cols = df.filter(regex='ipf').columns

# 2. Extract values as NumPy arrays for maximum speed
control_arr = df[control_cols].to_numpy()
ipf_arr = df[ipf_cols].to_numpy()

# 3. Run ks_2samp once per row using a list comprehension with zip
# (ks_2samp returns an object where you can access .statistic and .pvalue)
ks_results = [ks_2samp(c, i) for c, i in zip(control_arr, ipf_arr)]

# 4. Assign the results back to the DataFrame efficiently
df['ks_statics'] = [r.statistic for r in ks_results]
df['ks_p_values'] = [r.pvalue for r in ks_results]

In [58]:
# epsilon = 1e-9
epsilon = 1
np.log2((ipf.mean() + epsilon) / (control.mean() + epsilon))

df['log2_fold_change'] = np.log2((df[ipf_cols].mean(axis=1) + epsilon) / (df[control_cols].mean(axis=1) + epsilon))

In [59]:
df['bonf_p_values'] = df['ks_p_values'] * len(df)

In [60]:
df.head()

,ipf_724,ipf_726,ipf_729,ipf_730,ipf_731,ipf_732,ipf_733,ipf_735,control_738,ipf_739,...,control_1109,control_1111,control_1112,control_1113,ipf_1125,ipf_1130,ks_statics,ks_p_values,log2_fold_change,bonf_p_values
0,26.202635,39.826077,32.456940,102.684442,23.479752,33.248302,107.388687,57.548092,58.371059,31.989574,...,27.895753,26.026071,27.463970,33.176158,31.235186,30.728696,0.436893,3.264129e-09,0.789429,3.622204e-05
2,73.700699,81.218962,57.376486,86.708178,60.758032,73.815608,71.331885,84.276923,71.374254,58.227831,...,87.733207,71.926136,81.124341,63.314206,67.798439,69.985906,0.213592,1.797074e-02,0.100992,1.994213e+02
3,5.092385,7.906916,5.691501,12.773973,6.187082,10.259118,9.348060,8.780315,8.599308,5.785348,...,6.729052,5.468513,9.049026,6.045838,7.136678,6.640523,0.106796,6.020115e-01,0.020988,6.680521e+03
5,107.217851,109.749446,131.865931,55.389072,157.614792,126.675090,61.202106,55.291304,71.426371,240.296150,...,72.869668,47.874805,60.737625,128.694326,82.707253,66.307577,0.398058,1.122904e-07,-0.660811,1.246086e-03
6,591.827722,440.200216,332.299070,382.550570,247.327476,348.403415,311.352277,230.756563,100.611901,195.817030,...,103.235579,153.751292,256.858537,79.051614,324.572198,403.607093,0.825243,1.136421e-35,1.280747,1.261087e-31


In [61]:
to_keep = (df['bonf_p_values'] < 0.05) & (abs(df['log2_fold_change']) >= 2)

In [62]:
sig_genes = df.loc[to_keep]
len(sig_genes)

79

In [63]:
print(f"Upregulated genes: {np.sum(sig_genes['log2_fold_change'] > 0)}")
print(f"Downregulated genes: {np.sum(sig_genes['log2_fold_change'] < 0)}")

Upregulated genes: 65
Downregulated genes: 14


In [64]:
genes[sig_genes[sig_genes['log2_fold_change'] > 0].index]

array(['PROM1', 'CYP24A1', 'CP', 'CDH3', 'COL17A1', 'ATP12A', 'FAT2',
       'SLC4A11', 'DERL3', 'VSIG1', 'CD79A', 'COMP', 'COL1A1', 'COL7A1',
       'FHL2', 'TSPAN1', 'MUC5B', 'SPP1', 'IL13RA2', 'COL10A1', 'BPIFB1',
       'PDLIM4', 'TNS4', 'FCRLA', 'JCHAIN', 'POSTN', 'ERN2', 'TMPRSS4',
       'STRA6', 'GPR87', 'FCRL5', 'MUC4', 'CXCL14', 'FAM83A', 'PLEKHS1',
       'SYT8', 'THY1', 'CXCL13', 'MS4A1', 'SCGB3A1', 'CTHRC1', 'CLDN2',
       'GREM1', 'UGT1A6', 'COL3A1', 'SPRR1A', 'MZB1', 'KRT15', 'LGALS7B',
       'B3GNT3', 'MUC16', 'FDCSP', 'ALDH1A3', 'COL14A1', 'PLA2G2A',
       'S100A2', 'BPIFA1', 'IGFL2', 'LGALS7', 'SERPINB4', 'DIO2', 'IGLL5',
       'AC136428.1', 'MSMB', 'PRSS2'], dtype=object)

In [ ]:
# sig_genes[sig_genes['log2_fold_change'] > 0].to_csv("../data/GSE150910/upregulated_genes.csv", index=False)
# sig_genes[sig_genes['log2_fold_change'] < 0].to_csv("../data/GSE150910/downregulated_genes.csv", index=False)